# P11 on Attar: hybrid net-pin weighting for timing-driven global placement, on superblue18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayChang987/Attar/blob/main/examples/colab/p11_attar_superblue18.ipynb)

Timing-driven global placement adds timing to the placer's objective, most often by **weighting**: wires on
critical paths get pulled harder. **P11** (Li, Yang, Ai, Zhu, *Analytically-Derived Hybrid Net-Pin Weighting for
Timing-Driven Global Placement*, DAC 2026) combines two weightings:

* DATE25's **pin pairs**: every driver-sink wire on a critical path gets a weight between 10 and 50, and the placer
  minimizes the weighted squared distances of those pairs;
* **net weights derived from a delay model**: linearizing the cell delays (their NLDM tables) and the wire delays
  (Elmore) around the current placement tells how much a unit of a net's length costs in path delay; that becomes
  the net's weight in the placer's wirelength.

This notebook builds the method on [Attar](https://github.com/RayChang987/Attar), piece by piece, and places the
745k-cell superblue18 (ICCAD 2015) with it.

A GPU runtime is needed (Runtime -> Change runtime type -> T4 GPU). The first run downloads the design (60 MB)
and compiles Attar's timing kernels (a few minutes, once per runtime).

## 0. Setup

In [ ]:
!pip install -q attar==1.0.5

In [ ]:
import gc
import time

import numpy as np
import torch

import attar
from attar import Evaluator, naive
from attar.timing.sta import nldm

assert torch.cuda.is_available(), 'use a GPU runtime: Runtime -> Change runtime type -> T4 GPU'
DEVICE = 'cuda'
print('attar', attar.__version__, 'on', torch.cuda.get_device_name())

## 1. The design and its critical paths

superblue18 comes with the contest's placement. We time it once, as it is, to look at what the method works from:
**critical paths**. `critical_paths()` returns the worst path into every violating endpoint in one call.

In [ ]:
d = attar.load('after_fp/iccad2015/superblue18', device=DEVICE)
print('superblue18: %d cells (%d movable), %d nets' % (len(d.names['cells']), int(d.movable.sum()), len(d.n_ptr) - 1))
res = d.timing()
sm = res.summary()
print('given placement: WNS %.1f ps, TNS %.4g ps, %d of %d endpoints violate setup'
      % (sm['wns'], sm['tns'], sm['setup_violations'], sm['endpoints']))
cp = res.critical_paths()
print('%d critical paths (one per violating endpoint), %d path steps in all' % (len(cp), len(cp.pin)))

A path is a list of pin transitions from a startpoint to an endpoint. Consecutive pins are joined either by a
**cell arc** (an input pin to an output pin of the same cell) or by a **wire** (a driver pin to a sink pin of the
same net): `cp.wire` marks the wire steps. The method weights the wires. Here is the start of the worst path:

In [ ]:
steps = cp.report(0)
a = int(cp.ptr[0])
for n, (name, rf, t) in enumerate(steps[:10]):
    kind = '' if n == 0 else ('wire' if bool(cp.wire[a + n]) else 'cell')
    print('  %-4s %-32s %s  arrival %9.1f ps' % (kind, name, rf, t))
print('  ... %d steps, slack %.1f ps' % (len(steps), float(cp.slack[0])))

## 2. The pin pairs (from DATE25)

P11 starts from DATE25's pin pairs (DATE 2025, as its code DATE25-TDP implements them). It keeps a weight for every **pin pair**
(driver i, sink j) that has appeared on a critical path:

* a pair seen for the first time gets the minimum weight, 10;
* every later critical path through it adds 0.2 x slack / WNS (so up to 0.2 per path, the most for the worst
  path), up to a maximum of 50.

The weights accumulate over the placement: a pair that was critical once keeps being pulled. The placer then adds

$$\beta \sum_{(i, j)} w_{ij} \left[(x_i - x_j)^2 + (y_i - y_j)^2\right]$$

to its objective. The class below keeps the pairs as sorted keys `i * P + j` and merges each update into them.

In [ ]:
class PinPairs:
    """DATE25-TDP's pin-pair weights (net_weighting_scheme.h, NetWeighting<PIN2PIN>)."""
    MIN, MAX, ACC = 10.0, 50.0, 0.2           # pin2pin_min_weight, pin2pin_max_weight, pin2pin_accumulate_weight

    def __init__(self, d):
        self.d, self.dev = d, d.device
        self.P = int(d.p_cell.numel())
        self.keys = torch.zeros(0, dtype=torch.long, device=self.dev)
        self.w = torch.zeros(0, device=self.dev)

    def update(self, cp):
        """One round of critical paths: every wire step's (driver, sink) pair gets its increment."""
        k = cp.path_of()
        if len(cp) == 0 or len(cp.pin) < 2:
            return
        same = (k[1:] == k[:-1]) & cp.wire[1:]                     # wire steps within a path
        idx = torch.arange(1, len(cp.pin), device=self.dev)[same]
        key = cp.pin[idx - 1] * self.P + cp.pin[idx]                # (driver, sink)
        wns = float(cp.slack[0]) if len(cp.slack) else -1.0
        wns = wns if wns < 0 else -1.0
        inc = self.ACC * (cp.slack[k[idx]] / wns).clamp(0.0, 1.0)
        uk, inv = torch.unique(key, return_inverse=True)
        tot = torch.zeros(len(uk), device=self.dev).index_add_(0, inv, inc)
        if len(self.keys) == 0:                                    # every pair is new
            self.keys, self.w = uk, torch.full((len(uk),), self.MIN, device=self.dev)
            return
        merged = torch.unique(torch.cat([self.keys, uk]))
        old_pos = torch.searchsorted(merged, self.keys)
        new_pos = torch.searchsorted(merged, uk)
        q = torch.zeros(len(merged), device=self.dev).index_add_(0, old_pos, self.w)
        seen = torch.zeros(len(merged), dtype=torch.bool, device=self.dev)
        seen[old_pos] = True
        q.index_add_(0, new_pos, torch.where(seen[new_pos], tot, torch.zeros_like(tot)))
        self.keys = merged
        self.w = torch.where(seen, q.clamp(max=self.MAX), torch.full_like(q, self.MIN))

    def pairs(self):
        i = torch.div(self.keys, self.P, rounding_mode='floor')
        return i, self.keys - i * self.P, self.w


pp = PinPairs(d)
pp.update(cp)
print('%d pin pairs after one update, weight %.0f each' % (len(pp.keys), float(pp.w[0])))

## 3. Net weights from a delay model

P11 asks what a wire step i -> j on a critical path costs per unit of length. With $a_j$ the sensitivity of the
next cell's delay to its input slew, $b_i$ the sensitivity of the driving cell's delay to its load (which is also
its drive resistance $R_i$), $v_i$ the sensitivity of the driver's output slew to its load, $c$ the wire
capacitance per µm and $K_s$ the slew constant of the library, the paper's eqs. 17-18 give the net's weight
increment

$$w_e \mathrel{+}= b_i c + \left[(1 + a_j K_s) R_i + a_j v_i\right] c$$

summed over all critical paths for $w_\text{TNS}(e)$ and over the worst path only for $w_\text{WNS}(e)$. The net's
weight in the placer's wirelength then becomes (eq. 19)

$$W_e = W_e^0 \left[1 + \beta \left(t_1 w_\text{TNS}(e) + t_2 w_\text{WNS}(e)\right)\right].$$

### 3.1 The sensitivities, from the NLDM tables

The cell delay and output slew are NLDM tables of (input slew, load). Attar's timer exposes the tables and the
operating point of every arc, so the derivatives are one autograd call through the same bilinear interpolation the
timer uses (`nldm`). $K_s = \ln(\text{slew upper} / \text{slew lower threshold})$ comes from the library too.

In [ ]:
def slew_constant(T):
    """K_s = ln(slew_upper / slew_lower): ln 4 for the 20-80 % slews of the ICCAD 2015 libraries."""
    thr = T.t_thr
    vl, vh = thr[..., 1], thr[..., 2]
    ok = (vh > vl) & (vl > 0) & torch.isfinite(vh)
    ratio = float((vh[ok] / vl[ok]).median()) if bool(ok.any()) else 4.0
    return float(np.log(ratio)) if ratio > 1 else float(np.log(4.0))


def arc_index(T, fr, to):
    """The timer's delay arc for each (from pin, to pin), -1 where there is none."""
    key = T.c_fr * T.P + T.c_to
    order = torch.argsort(key)
    skey = key[order]
    q = fr * T.P + to
    pos = torch.searchsorted(skey, q).clamp(max=len(skey) - 1)
    return torch.where(skey[pos] == q, order[pos], torch.full_like(q, -1))


def sensitivities(T, arcs, orf, irf):
    """(a, b, v) per arc at its operating point: dDelay/dSlew_in, dDelay/dLoad, dSlew_out/dLoad. arcs < 0: zeros."""
    live = arcs >= 0
    a0 = torch.where(live, arcs, torch.zeros_like(arcs))
    orf, irf = orf.clamp(0, 1), irf.clamp(0, 1)
    uk, inv = torch.unique(a0 * 4 + orf * 2 + irf, return_inverse=True)          # each (arc, transitions) once
    ua, ur, ui = uk // 4, (uk % 4) // 2, uk % 2
    lib = T.c_lib[ua]
    clean = lambda t: torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)       # noqa: E731
    x = clean(T.smax[T.c_fr[ua], ui]).clamp(min=0.0).clone().requires_grad_(True)   # input slew
    y = clean(T.load[T.c_to[ua], ur]).clamp(min=0.0).clone().requires_grad_(True)   # load
    table = lambda slot: nldm(T.t_ix1[lib, slot], T.t_ix2[lib, slot], T.t_val[lib, slot], T.t_n[lib, slot], x, y)  # noqa: E731
    delay, slew = table(ur), table(2 + ur)                     # slot orf: delay, 2 + orf: output slew
    a, b = torch.autograd.grad(delay.sum(), (x, y), retain_graph=True)
    _, v = torch.autograd.grad(slew.sum(), (x, y))
    zero = torch.zeros((), device=arcs.device)
    return [torch.where(live, clean(t)[inv], zero) for t in (a, b, v)]

### 3.2 Accumulating the net weights

For every wire step i -> j of the critical paths: the cell arc that arrives at the driver i gives $b_i = R_i$ and
$v_i$, the cell arc that leaves the sink j gives $a_j$. The increments go to the step's net, for every path
(TNS) and for the worst path (WNS).

In [ ]:
class NetWeights:
    """P11's derived net weights w_TNS(e), w_WNS(e) (paper eqs. 17-18), cumulative."""

    def __init__(self, d):
        self.d = d
        E = len(d.n_weight)
        self.tns = torch.zeros(E, device=d.device)
        self.wns = torch.zeros(E, device=d.device)

    def update(self, res, cp):
        T = res.timer
        ks, c = slew_constant(T), float(T.c_sig)
        k = cp.path_of()
        R = len(cp.pin)
        if len(cp) == 0 or R < 2:
            return
        pin, rf, wire, ptr = cp.pin, cp.rf, cp.wire, cp.ptr
        same = (k[1:] == k[:-1]) & wire[1:]
        idx = torch.arange(1, R, device=pin.device)[same]          # row of the sink j of each wire step
        kk = k[idx]
        prev_cell = ((idx - 1) > ptr[kk]) & ~wire[idx - 1]           # a cell arc arrives at the driver i
        nxt = (idx + 1).clamp(max=R - 1)
        next_cell = ((idx + 1) < ptr[kk + 1]) & ~wire[nxt]           # a cell arc leaves the sink j
        i, j = pin[idx - 1], pin[idx]
        before, after = (idx - 2).clamp(min=0), nxt
        prev_arc = arc_index(T, pin[before], i)
        next_arc = arc_index(T, j, pin[after])
        _, b_i, v_i = sensitivities(T, torch.where(prev_cell, prev_arc, -1), rf[idx - 1], rf[before])
        a_j, _, _ = sensitivities(T, torch.where(next_cell, next_arc, -1), rf[after], rf[idx])
        R_i = b_i
        w = b_i * c + ((1.0 + a_j * ks) * R_i + a_j * v_i) * c
        net = self.d.p_net[i]
        self.tns.index_add_(0, net, w)
        self.wns.index_add_(0, net, torch.where(kk == 0, w, torch.zeros((), device=w.device)))

    def factor(self, base, beta, t1, t2):
        """Eq. 19: the nets' weights for the placer."""
        return base * (1.0 + beta * (t1 * self.tns + t2 * self.wns))


nw = NetWeights(d)
nw.update(res, cp)
top = torch.topk(nw.tns, 5).indices
print('%d nets on critical paths; the five heaviest, w_TNS in ps/um:' % int((nw.tns > 0).sum()))
for e in top.tolist():
    print('  net %-24s w_TNS %8.3f   w_WNS %6.3f' % (d.names['nets'][e], float(nw.tns[e]), float(nw.wns[e])))

## 4. Into the placer

Attar's global placer (`naive.global_place`, electrostatic, as DREAMPlace) has two hooks, and the method uses both:

* `extra(pos) -> (value, gradient)`: a term added to the objective; the pin-pair term goes here;
* `callback(it, pos, info)`: called after every iteration; here it times the placement every few iterations,
  updates both weightings and writes the net weights into `d.n_weight`, which the placer reads from the next
  iteration on.

### 4.1 The pin-pair term

In [ ]:
def pin_pair_term(d, pairs, beta):
    """extra(pos) for global_place: beta * sum w_ij * |p_i - p_j|^2 over the pin pairs, and its gradient."""
    cache = {}

    def extra(pos):
        i, j, w = pairs.pairs()
        if len(i) == 0:
            return 0.0, torch.zeros_like(pos)
        if cache.get('keys') is not pairs.keys:                  # new pairs: the pins they touch
            sel, inv = torch.unique(torch.cat([i, j]), return_inverse=True)
            cache.update(keys=pairs.keys, sel=sel, ii=inv[:len(i)], jj=inv[len(i):])
        sel, ii, jj = cache['sel'], cache['ii'], cache['jj']
        pos = pos.detach().requires_grad_(True)
        p = pos[d.p_cell[sel]] + d.size[d.p_cell[sel]] * 0.5 + d.pin_off[sel]   # pin positions
        dx, dy = p[ii, 0] - p[jj, 0], p[ii, 1] - p[jj, 1]
        v = (w * (dx * dx + dy * dy)).sum() * beta
        g, = torch.autograd.grad(v, pos)
        return float(v.detach()), g

    return extra

### 4.2 The whole method

The paper's schedule: place for wirelength until the cells have spread (density overflow 0.55, about iteration
355 here, the paper's "iteration 500 of DREAMPlace"), then every 15 iterations time the placement, update the
weights and go on. The paper's parameters: β = 2.5e-5, t1 = 3e5, t2 = 1e4.

β is in DREAMPlace's units, where lengths are counted in placement sites (s µm each). In Attar's µm: the pin-pair
weights are unit-free and multiply squared distances, so the pair term's β becomes β / s; the net weights are
delays per length, so the net factor's β becomes β · s, with the weights read in ns (x 1e-3) as DREAMPlace's
timer reports them.

After global placement, Attar's legalizer (Abacus) puts the cells on rows, and Attar's timer (equal to OpenSTA)
measures the result.

In [ ]:
BETA, T1, T2 = 2.5e-5, 3e5, 1e4
START_OVERFLOW, EVERY = 0.55, 15


def place(seed=0, verbose=True):
    """P11: global placement with the pin-pair term and the derived net weights, then legalization and timing. Returns (metrics, trace)."""
    d = attar.load('after_fp/iccad2015/superblue18', device=DEVICE)
    d.unplace()                                                # global placement from scratch
    base = d.n_weight.clone()
    s = float(d.row_site_w.float().median())                    # site width (um)
    pairs = PinPairs(d)
    nets = NetWeights(d)
    state = {'start': None}
    trace = []

    def callback(it, pos, info):
        if state['start'] is None:                               # wait until the cells have spread
            if it % 5 or info['overflow'] > START_OVERFLOW:
                return False
            state['start'] = it
        if (it - state['start']) % EVERY:
            return False
        t0 = time.time()
        res = d.timing(pos)                                      # time the (unlegalized) placement
        cp = res.critical_paths()
        pairs.update(cp)
        nets.update(res, cp)
        d.n_weight.copy_(nets.factor(base, BETA * s * 1e-3, T1, T2))
        trace.append({'it': it, 'tns': float(res.tns), 'wns': float(res.wns), 'overflow': info['overflow']})
        if verbose:
            print('  it %4d  overflow %.3f  TNS %.4g  WNS %.4g  pairs %d  (%.1fs)'
                  % (it, info['overflow'], res.tns, res.wns, len(pairs.keys), time.time() - t0), flush=True)
        return False

    t_start = time.time()
    gp = naive.global_place(d, seed=seed, max_iters=2500, stop_overflow=0.07, bins=512, verbose=False,
                            extra=pin_pair_term(d, pairs, BETA / s), callback=callback)
    d.n_weight.copy_(base)                                       # measure with the netlist's own weights
    pre = d.timing()
    for _ in range(3):                                           # legalize (a second pass repairs leftovers)
        naive.legalize(d, verbose=False)
        if Evaluator(d).legality()['legal']:
            break
    r = d.timing()
    m = {'legal': bool(Evaluator(d).legality()['legal']), 'hpwl_um': float(d.hpwl()), 'tns': float(r.tns),
         'wns': float(r.wns), 'tns_before_lg': float(pre.tns), 'wns_before_lg': float(pre.wns),
         'gp_iterations': gp['iterations'], 'timing_from': state['start'], 'seconds': time.time() - t_start}
    return m, trace

## 5. Place superblue18

The design loaded in section 1 is released first, so that a 16 GB T4 has room for the placement.

In [ ]:
del d, res, cp, pp, nw
gc.collect(); torch.cuda.empty_cache()

SEED = 0
m, trace = place(SEED)
print('\nP11, seed %d: TNS %.4g ps  WNS %.1f ps  HPWL %.4g um  legal %s  (%.0f s)'
      % (SEED, m['tns'], m['wns'], m['hpwl_um'], m['legal'], m['seconds']))
print('before legalization: TNS %.4g ps  WNS %.1f ps' % (m['tns_before_lg'], m['wns_before_lg']))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r['it'] for r in trace], [r['tns'] for r in trace], color='#2a6fdb')
ax[1].plot([r['it'] for r in trace], [r['wns'] for r in trace], color='#2a6fdb')
for a_, name in zip(ax, ('TNS', 'WNS')):
    a_.set_xlabel('global placement iteration'); a_.set_ylabel('%s before legalization (ps)' % name); a_.grid(alpha=.3)
ax[0].set_title('P11: timing while placing (every %d iterations)' % EVERY)
plt.tight_layout(); plt.show()

## Notes

* **Our runs** of this notebook's code (superblue18, seeds 0-2, RTX 4090): TNS -2.57e6 to -2.60e6 ps, WNS about
  -7.7e3 ps after legalization.
* **Which timer.** Attar's timer equals OpenSTA (about 1 ps per endpoint on superblue). The papers and DREAMPlace
  report OpenTimer, which treats some input / output paths of these designs differently, so the absolute TNS here
  is larger and **not comparable with the papers' tables**.
* **Run to run.** GPU floating-point sums are not ordered, and timing feedback amplifies the difference, so a run
  with the same seed can end a few percent elsewhere. `naive.global_place(..., deterministic=True)` makes GPU runs
  repeatable (slower); try several seeds (`SEED`).